In [1]:
from datasets import load_dataset
import pandas as pd
import pickle

# 加载 Qilin 数据集
src_train = load_dataset("THUIR/Qilin", "search_train")["train"].to_pandas()
rec_train = load_dataset("THUIR/Qilin", "recommendation_train")["train"].to_pandas()
src_test = load_dataset("THUIR/Qilin", "search_test")["train"].to_pandas()
rec_test = load_dataset("THUIR/Qilin", "recommendation_test")["train"].to_pandas()
note_info = load_dataset("THUIR/Qilin", "notes")["train"].to_pandas()


In [11]:
import pandas as pd
import numpy as np

# 1. 计算每个用户的点击总量
def count_clicks(df, detail_col):
    exploded = df.explode(detail_col)
    exploded['is_click'] = exploded[detail_col].apply(
        lambda x: 1 if isinstance(x, dict) and x.get('click') == 1 else 0
    )
    return exploded.groupby('user_idx')['is_click'].sum()

user_rec_click_total = count_clicks(pd.concat([rec_train, rec_test]), 'rec_result_details_with_idx')
user_src_click_total = count_clicks(pd.concat([src_train, src_test]), 'search_result_details_with_idx')

# 2. 计算“距离” 30/20 最近的 5 个人
user_metrics = pd.DataFrame({'rec_c': user_rec_click_total, 'src_c': user_src_click_total}).fillna(0)

# 计算每个用户到 (30, 20) 的距离： sqrt((rec-30)^2 + (src-20)^2)
user_metrics['distance'] = np.sqrt((user_metrics['rec_c'] - 30)**2 + (user_metrics['src_c'] - 20)**2)

# 选取距离最近的 5 个用户
selected_users = user_metrics.sort_values('distance').head(10).index.tolist()

# 3. 提取历史并按各自的时间戳排序
def get_user_timeline_final(df, detail_col, is_src=False):
    subset = df[df['user_idx'].isin(selected_users)].copy()
    exploded = subset.explode(detail_col)
    
    # 动态识别时间戳字段
    ts_key = 'search_timestamp' if is_src else 'request_timestamp'
    
    # 提取信息
    exploded['note_idx'] = exploded[detail_col].apply(lambda x: x.get('note_idx') if isinstance(x, dict) else None)
    exploded['click_val'] = exploded[detail_col].apply(lambda x: x.get('click') if isinstance(x, dict) else 0)
    exploded['ts'] = exploded[detail_col].apply(lambda x: x.get(ts_key) if isinstance(x, dict) else None)
    
    # 关联标题
    merged = exploded.merge(note_info[['note_idx', 'note_title']], on='note_idx', how='left')
    
    # 标注场景，仅保留点击记录
    tag = "!src: " if is_src else ""
    merged['formatted_title'] = merged.apply(
        lambda row: f"{tag}{row['note_title']}" if pd.notnull(row['note_title']) and row['click_val'] == 1 else None, 
        axis=1
    )
    
    return merged[['user_idx', 'ts', 'formatted_title']].dropna(subset=['formatted_title'])

# 4. 合流、转换时间并排序输出
src_flow = get_user_timeline_final(pd.concat([src_train, src_test]), 'search_result_details_with_idx', is_src=True)
rec_flow = get_user_timeline_final(pd.concat([rec_train, rec_test]), 'rec_result_details_with_idx', is_src=False)

full_timeline = pd.concat([src_flow, rec_flow])
full_timeline['dt'] = pd.to_datetime(full_timeline['ts'], unit='s')
full_timeline = full_timeline.sort_values(by=['user_idx', 'dt'])

# 5. 打印报告
for uid in selected_users:
    m = user_metrics.loc[uid]
    print(f"=== 典型用户: {uid} (实际点击 - 推荐:{int(m.rec_c)}, 搜索:{int(m.src_c)}) ===")
    user_data = full_timeline[full_timeline['user_idx'] == uid]
    for _, row in user_data.iterrows():
        print(f"[{row['dt']}] {row['formatted_title']}")
    print("\n" + "="*80 + "\n")

=== 典型用户: 5214 (实际点击 - 推荐:30, 搜索:20) ===
[2024-11-27 01:19:00] 天哪，N下海了
[2024-11-27 01:19:00] Infinite没人抢，嘻嘻，过气了吗😂
[2024-11-27 01:19:00] 明星都来长沙润府看房了……
[2024-11-27 01:19:12] 我勒个清汤大老爷！一觉醒来天塌了又塌😭
[2024-11-27 01:20:30] nan
[2024-11-27 01:22:45] 你以为养生，其实伤身的行为！
[2024-11-27 04:37:24] A-Lin11.30长沙演唱会
[2024-11-27 04:37:24] 血型不同，睡眠状况为何大不同？
[2024-11-27 10:11:37] 你们的山姆没有这个抽奖入口吗？抽中鳕鱼🐟
[2024-11-27 10:11:37] 破纪录了❗小女孩挑战吃完10斤巨肥牛肉
[2024-11-27 10:11:37] 《恰饭哒咧》EP1：请叫我牛肋条皇后！
[2024-11-27 10:34:08] 长沙终于也有了😭😭全场9.9r简直🔪挲疯了！！
[2024-11-27 10:34:08] 没课回村 宅家都吃了啥
[2024-11-27 10:34:08] 山姆毒山楂
[2024-11-27 10:35:42] 演唱会带镜子有一种跟爱豆一起上大屏的错觉
[2024-11-27 12:23:17] nan
[2024-11-27 12:23:17] 女明星也拒绝不了的巧克力零食？浅测一波‼️
[2024-11-27 12:23:17] 来了来了！羊毛月选秀舞台视频
[2024-11-27 12:29:27] 来揭秘一下少爷的下午茶！知己知彼百战不殆
[2024-11-27 12:34:36] 再闯麓山南路美食，吃到最后小两口直摇头
[2024-11-27 12:34:36] nan
[2024-11-27 12:34:36] 这是本美食博主测山姆新品最满意的一次！！！
[2024-11-27 12:37:52] !src: 生完孩子以后，经量越来越少，你也是这样吗？
[2024-11-27 12:38:24] 一万八拿下冰飘花手镯，妈妈给女儿的入学礼
[2024-11-27 12:41:03] 看过的都估计当爷爷奶奶了
[